In [7]:
import sdv
print(sdv.__version__)

1.34.3


In [9]:
import pandas as pd
import itertools
from sdv.single_table import CTGANSynthesizer
# from sdv.metadata import SingleTableMetadata # class SingleTableMetadata is deprecated
from sdv.metadata import Metadata
import numpy as np
import pickle

# Load the original data
## original_data = pd.read_csv('Boston.csv')
original_data = pd.read_excel("/Users/soupornoghosh/Desktop/IMT 600/Synthetic Data/Arquitectura y Concreto_translated.xlsx")
original_data.head()

# Strip all leading/trailing spaces from column names right after loading the data
original_data.columns = original_data.columns.str.strip()

# Check
print("Original Data:")
print(original_data.head())

# Check the metadata/data structure
# metadata = SingleTableMetadata() # class SingleTableMetadata is deprecated
# metadata.detect_from_dataframe(original_data)

metadata = Metadata.detect_from_dataframe(data=original_data, table_name='employees') # we can use any string name. It's just a label for SDV to internally reference the table.

# PII handling matters! We need to flag E-mail, First Name, and Yam as PII so SDV uses its anonymization engine (powered by Faker)
metadata.update_column(table_name='employees', column_name='Yam', sdtype='name', pii=True)
metadata.update_column(table_name='employees', column_name='First Name', sdtype='first_name', pii=True)
metadata.update_column(table_name='employees', column_name='E-mail', sdtype='email', pii=True)

print(metadata)

# Define batch sizes and epochs to test
batch_sizes = [200, 300, 500]
epochs_list = [1000, 2000, 3000, 5000]
batch_epoch_combinations = list(itertools.product(batch_sizes, epochs_list))

# Function to evaluate the synthetic data quality
def evaluate_synthetic_data(original_data, synthetic_data):
    # Example evaluation using mean absolute error
    errors = []
    for col in original_data.columns:
        if original_data[col].dtype in [np.float64, np.int64]:
            error = np.mean(np.abs(original_data[col] - synthetic_data[col]))
            errors.append(error)
    return np.mean(errors)

best_model = None
best_score = float('inf')
best_params = None

# Loop over each combination of batch size and epochs
for batch_size, epochs in batch_epoch_combinations:
    try:
        print(f"Training model with batch_size={batch_size}, epochs={epochs}")
        
        synthesizer = CTGANSynthesizer(
            metadata, 
            enforce_min_max_values=True,
            enforce_rounding=True,
            batch_size=batch_size, 
            epochs=epochs, 
            verbose=True,
            #cuda=False  # Set to False if CUDA is not available # This will fail on Mac — Macs don't have NVIDIA GPUs
            enable_gpu=False # cuda parameter is deprecated
        )
        
        # Fit the synthesizer to the original data and generate synthetic data
        synthesizer.fit(original_data)
        synthetic_data = synthesizer.sample(len(original_data))

        # Evaluate the synthetic data
        score = evaluate_synthetic_data(original_data, synthetic_data)
        print(f"Score for batch_size={batch_size}, epochs={epochs}: {score}")
        
        # Check if this model is the best so far
        if score < best_score:
            best_score = score
            best_model = synthesizer
            best_params = (batch_size, epochs)
    except Exception as e:
        print(f"An error occurred for batch_size={batch_size}, epochs={epochs}: {e}")

# Save the best model
if best_model is not None:
    best_model_filename = f"best_model_bs{best_params[0]}_ep{best_params[1]}.pkl"
    with open(best_model_filename, 'wb') as f:
        pickle.dump(best_model, f)
    print(f"Best model saved as {best_model_filename}")

    # Save the synthetic data from the best model
    synthetic_data = best_model.sample(len(original_data))
    synthetic_data.to_csv('synthetic_data.csv', index=False)
    print("Best synthetic data generated and saved to 'synthetic_data.csv'")
else:
    print("No valid model was trained.")


Original Data:
                                 Yam  \
0           JORGE MARIO OSPINA PARRA   
1       MARIA ISABEL GALLEGO PALACIO   
2        JORGE HERNAN RAMIREZ PELAEZ   
3     GLORIA CRISTINA OROZCO SALAZAR   
4   GERMAN EDUARDO MARTINEZ ARBOLEDA   

                                      E-mail        First Name  \
0      jorgeospina@arquitecturayconcreto.com       JORGE MARIO   
1    mariaigallego@arquitecturayconcreto.com      MARIA ISABEL   
2     jorgeramirez@arquitecturayconcreto.com      JORGE HERNAN   
3   cristinaorozco@arquitecturayconcreto.com   GLORIA CRISTINA   
4   germanmartinez@arquitecturayconcreto.com    GERMAN EDUARDO   

   Years of Service                        AREA              AREA 2  \
0                22                   Machinery   Construction site   
1                22   Administrative Management      Administrative   
2                 8        Financial Management      Administrative   
3                12        Technical Management      Administra

/opt/anaconda3/envs/vsynth/lib/python3.11/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.30) | Discrim. (-00.21): 100%|████████████████████████████████████████████████████| 1000/1000 [02:29<00:00,  6.71it/s]


Score for batch_size=200, epochs=1000: 6.13293144634223
Training model with batch_size=200, epochs=2000


Gen. (-00.53) | Discrim. (+00.28): 100%|████████████████████████████████████████████████████| 2000/2000 [05:02<00:00,  6.62it/s]


Score for batch_size=200, epochs=2000: 5.860076040955045
Training model with batch_size=200, epochs=3000


...

Score for batch_size=500, epochs=2000: 6.354892260855424
Training model with batch_size=500, epochs=3000


Gen. (-00.84) | Discrim. (-00.11): 100%|████████████████████████████████████████████████████| 3000/3000 [03:36<00:00, 13.85it/s]


Score for batch_size=500, epochs=3000: 5.737721863272731
Training model with batch_size=500, epochs=5000


Gen. (-00.66) | Discrim. (-00.08): 100%|████████████████████████████████████████████████████| 5000/5000 [06:09<00:00, 13.54it/s]


Score for batch_size=500, epochs=5000: 5.949487744773468
Best model saved as best_model_bs500_ep3000.pkl
Best synthetic data generated and saved to 'synthetic_data.csv'


In [10]:
import os
print(os.getcwd())

/Users/soupornoghosh/SyntheticData_OrganizationalScience/Demo


In [11]:
#### Evaluation
from sdv.single_table import CTGANSynthesizer
# from sdv.metadata import SingleTableMetadata # SingleTableMetadata is deprecated
from sdv.metadata import Metadata
from sdv.evaluation.single_table import run_diagnostic
from sdv.evaluation.single_table import evaluate_quality
from sdv.evaluation.single_table import get_column_pair_plot
import pandas as pd

# Here, we do some evaluation to see how well the synthetic data matches the original data
original_data = pd.read_excel("/Users/soupornoghosh/Desktop/IMT 600/Synthetic Data/Arquitectura y Concreto_translated.xlsx")
original_data.columns = original_data.columns.str.strip()  # strip spaces like before
synthetic_data = pd.read_csv('synthetic_data.csv')

original_data.keys()

# Detect metadata
# metadata = SingleTableMetadata()
# metadata.detect_from_dataframe(original_data)
metadata = Metadata.detect_from_dataframe(data=original_data, table_name='employees')

# PII corrections — same as before
metadata.update_column(table_name='employees', column_name='Yam', sdtype='name', pii=True)
metadata.update_column(table_name='employees', column_name='First Name', sdtype='first_name', pii=True)
metadata.update_column(table_name='employees', column_name='E-mail', sdtype='email', pii=True)

# Here is check the data types of the columns and data structure.
diagnostic = run_diagnostic(
    real_data=original_data,
    synthetic_data=synthetic_data,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 2374.09it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 214.78it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [12]:
# Here we check how close the two data is. Score close to 1 is good.
quality_report = evaluate_quality(
    original_data,
    synthetic_data,
    metadata
)

quality_report.get_details(property_name='Column Shapes')

Generating report ...

(1/2) Evaluating Column Shapes: |████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 1096.25it/s]|
Column Shapes Score: 86.97%

(2/2) Evaluating Column Pair Trends: |█████████████████████████████████████████████████████| 190/190 [00:00<00:00, 1182.72it/s]|
Column Pair Trends Score: 73.64%

Overall Score (Average): 80.31%



,Column,Metric,Score
0,Years of Service,KSComplement,0.847765
1,AREA,TVComplement,0.892458
2,AREA 2,TVComplement,0.934358
3,Communication - It works,TVComplement,0.529485
4,CulturaAyC - We add value to clients (internal...,TVComplement,0.966770
5,CulturaAyC - We genuinely care for people,TVComplement,0.870163
6,CulturaAyC - We execute with discipline,TVComplement,0.933842
7,CulturaAyC - We promote high-performing teams,TVComplement,0.846425
8,"CulturaAyC - We innovate, we learn and we adapt",TVComplement,0.964946
9,PERFORMANCE,KSComplement,0.772346


In [13]:
# Here we check how can the privacy be reserved in the synthetic data

# Find overlapping rows
overlapping_data = pd.merge(original_data, synthetic_data, how='inner')

# Display the overlapping rows
print("Overlapping data:")
print(overlapping_data)

Overlapping data:
Empty DataFrame
Columns: [Yam, E-mail, First Name, Years of Service, AREA, AREA 2, CITY, Communication - It works, CulturaAyC - We add value to clients (internal and external), CulturaAyC - We genuinely care for people, CulturaAyC - We execute with discipline, CulturaAyC - We promote high-performing teams, CulturaAyC - We innovate, we learn and we adapt, PERFORMANCE, AGE, GENDER, List, JOB LEVEL, NPS company, Status]
Index: []


In [14]:
# Here we show the distribution of selected columns in the original and synthetic data

fig = get_column_pair_plot(
    real_data=original_data,
    synthetic_data=synthetic_data,
    metadata=metadata,
    column_names=['PERFORMANCE', 'Years of Service'],
    )
    
fig.show()